# JAX: Functional Arrays

**Interview focus:** immutable arrays, PRNG keys, functional style, and differences from NumPy.

In [ ]:
import jax
import jax.numpy as jnp
from jax import random

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

## 1. JAX Arrays vs NumPy

JAX arrays are **immutable**. In-place updates are not allowed.

In [ ]:
import numpy as np

x_np = np.array([1, 2, 3])
x_np[0] = 99  # OK in NumPy

x_jax = jnp.array([1, 2, 3])
# x_jax[0] = 99  # ERROR — JAX arrays are immutable
x_jax = x_jax.at[0].set(99)  # functional update — returns new array
print(f"original unchanged conceptually, new array: {x_jax}")

## 2. Array Creation (NumPy-compatible API)

In [ ]:
zeros = jnp.zeros((3, 4))
ones = jnp.ones((2, 3))
arange = jnp.arange(0, 10, 2)
eye = jnp.eye(3)

# From NumPy
np_arr = np.array([1.0, 2.0, 3.0])
jnp_arr = jnp.array(np_arr)

print(zeros.shape, zeros.dtype)

## 3. PRNG Keys — JAX's Random Number System

**Critical interview topic:** JAX requires explicit PRNG key splitting. Never reuse keys.

In [ ]:
key = random.PRNGKey(42)
print(f"key: {key}")

# Split key before each random operation
key, subkey = random.split(key)
samples = random.normal(subkey, shape=(5,))
print(f"samples: {samples}")

# Split into multiple subkeys
key, *subkeys = random.split(key, num=4)
for i, sk in enumerate(subkeys):
  print(f"  subkey {i}: {random.normal(sk, shape=(2,))}")

## 4. Device Placement

In [ ]:
x = jnp.ones(3)
print(f"default device: {x.devices()}")

# Explicit device placement
with jax.default_device(jax.devices()[0]):
  y = jnp.zeros(3)
  print(f"placed on: {y.devices()}")

## 5. Functional Updates with `.at[]`

In [ ]:
x = jnp.zeros(5)

x = x.at[2].set(10)           # set index 2 to 10
x = x.at[0:2].add(1)          # add 1 to indices 0, 1
x = x.at[3].multiply(5)       # multiply index 3 by 5
x = x.at[jnp.array([1, 3])].set(-1)  # fancy indexing

print(x)

## 6. Pure Functions — The JAX Way

JAX functions should be **pure**: no side effects, same inputs → same outputs.

In [ ]:
def normalize(x):
  """Pure function — no mutation, no global state."""
  mean = x.mean()
  std = x.std()
  return (x - mean) / (std + 1e-8)

data = random.normal(random.PRNGKey(0), shape=(100,))
normalized = normalize(data)
print(f"mean: {normalized.mean():.2e}, std: {normalized.std():.4f}")

## 7. Practice Problems

In [ ]:
# Problem: Generate a batch of random data with unique subkeys
def generate_batch(key, batch_size, feature_dim):
  keys = random.split(key, batch_size)
  return jax.vmap(lambda k: random.normal(k, shape=(feature_dim,)))(keys)

key = random.PRNGKey(0)
batch = generate_batch(key, 8, 4)
print(f"batch shape: {batch.shape}")

# Problem: Implement softmax in JAX
def softmax(x):
  x = x - x.max(axis=-1, keepdims=True)
  exp_x = jnp.exp(x)
  return exp_x / exp_x.sum(axis=-1, keepdims=True)

logits = jnp.array([[1.0, 2.0, 3.0], [1.0, 1.0, 1.0]])
print(softmax(logits))